# Análisis Descriptivo de Micronegocios en Cesar
**Fuente:** DANE – Encuesta de Micronegocios (EMICRON) – Módulo características del micronegocio
**Departamento:** Cesar (COD_DEPTO = 20)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})

RUTA = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
        r'\Documentos\2025\Adapta Cesar\micronegocios_cesar.csv')
df_raw = pd.read_csv(RUTA, encoding='latin-1', low_memory=False)
print(f'Filas totales: {len(df_raw):,}  |  Columnas: {df_raw.shape[1]}')
cesar = df_raw[df_raw['COD_DEPTO'] == 20].copy()
universo = cesar['F_EXP'].sum()
print(f'Registros muestra Cesar: {len(cesar):,}')
print(f'Universo estimado de micronegocios en Cesar: {universo:,.0f}')

## 1. Diccionario de variables (Módulo características del micronegocio)

In [ ]:
diccionario = {
    'P1633':    'Registro en RUT (1=Sí, 2=No)',
    'P986':     'Régimen tributario (1=Simplificado, 2=Común/ordinario, 9=No sabe)',
    'P640':     'Lleva registros contables (1=Sí, 2=No)',
    'P4000':    'Razón principal para no llevar registro contable (1-8)',
    'P1055':    'Matriculado en Cámara de Comercio (1=Sí, 2=No)',
    'P1056':    'Año de matrícula en Cámara de Comercio',
    'P661':     'Matrícula mercantil renovada (1=Sí, 2=No)',
    'P1057':    'Razón principal para no estar matriculado (1-8)',
    'P4004':    'Año de vencimiento de la matrícula',
    'P2991':    'Presentó declaración de renta último año (1=Sí, 2=No)',
    'P2992':    'Presentó declaración de IVA último año (1=Sí, 2=No)',
    'P2993':    'Presentó declaración de ICA último año (1=Sí, 2=No)',
    'CLASE_TE': 'Tipo establecimiento (1=Vivienda,2=Local,3=Vía pública,4=Obra,5=Vehículo,6=Otro)',
    'AREA':     'Área geográfica (1=Cabecera, 2=Centros poblados, 3=Rural disperso)',
    'F_EXP':    'Factor de expansión (peso muestral)'
}
pd.DataFrame.from_dict(diccionario, orient='index', columns=['Descripción']).rename_axis('Variable')

## 2. Distribución por zona geográfica (AREA)

In [ ]:
zona_map = {1: 'Cabecera', 2: 'Centros poblados', 3: 'Rural disperso'}
zona = cesar.groupby('AREA')['F_EXP'].sum().rename(index=zona_map)
zona_pct = (zona / zona.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(zona.index, zona.values, color=['#2196F3', '#4CAF50', '#FF9800'])
ax.bar_label(bars, labels=[f'{v:,.0f} ({p}%)' for v, p in zip(zona.values, zona_pct.values)], padding=5)
ax.set_xlabel('Micronegocios estimados')
ax.set_title('Micronegocios por zona geográfica – Cesar')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout(); plt.show()
print(zona.to_frame('Universo').assign(Pct=zona_pct))

## 3. Formalización tributaria – RUT (P1633) y Régimen (P986)

In [ ]:
rut_map = {1: 'Sí tiene RUT', 2: 'No tiene RUT'}
reg_map = {1: 'Simplificado', 2: 'Común/Ordinario', 9: 'No sabe'}
rut = cesar.groupby('P1633')['F_EXP'].sum().rename(index=rut_map)
reg = cesar[cesar['P1633'] == 1].groupby('P986')['F_EXP'].sum().rename(index=reg_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].pie(rut.values, labels=rut.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Registro en RUT (P1633)')
axes[1].pie(reg.values, labels=reg.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF9800', '#9E9E9E'], startangle=90)
axes[1].set_title('Régimen tributario con RUT (P986)')
plt.suptitle('Formalización tributaria – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('RUT:', rut.to_frame('Universo').assign(Pct=(rut/rut.sum()*100).round(1)))
print('\nRégimen (con RUT):', reg.to_frame('Universo').assign(Pct=(reg/reg.sum()*100).round(1)))

## 4. Registros contables (P640) y razón para no llevarlos (P4000)

In [ ]:
cont_map = {1: 'Lleva registros', 2: 'No lleva registros'}
razon_cont_map = {1:'No lo considera necesario',2:'No sabe cómo',3:'No tiene tiempo',
                  4:'Es muy costoso',5:'No lo exigen',6:'Negocio muy pequeño',
                  7:'Lo hace mentalmente',8:'Otro'}
cont = cesar.groupby('P640')['F_EXP'].sum().rename(index=cont_map)
razon_no = cesar[cesar['P640'] == 2].groupby('P4000')['F_EXP'].sum().rename(index=razon_cont_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(cont.values, labels=cont.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('¿Lleva registros contables? (P640)')
rs = razon_no.sort_values(ascending=True)
axes[1].barh(rs.index, rs.values, color='#FF9800')
axes[1].bar_label(axes[1].containers[0], labels=[f'{v:,.0f}' for v in rs.values], padding=4)
axes[1].set_xlabel('Micronegocios estimados')
axes[1].set_title('Razón para no llevar contabilidad (P4000)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('Registros contables – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Cámara de Comercio (P1055, P661, P1057)

In [ ]:
camara_map = {1: 'Matriculado', 2: 'No matriculado'}
renov_map  = {1: 'Renovada', 2: 'No renovada'}
razon_cam_map = {1:'No lo considera necesario',2:'Es muy costoso',3:'No sabe cómo',
                 4:'No lo exigen',5:'Trámite difícil',6:'No tiene tiempo',
                 7:'Negocio muy pequeño',8:'Otro'}
camara = cesar.groupby('P1055')['F_EXP'].sum().rename(index=camara_map)
renov  = cesar[cesar['P1055'] == 1].groupby('P661')['F_EXP'].sum().rename(index=renov_map)
razon_cam = cesar[cesar['P1055'] == 2].groupby('P1057')['F_EXP'].sum().rename(index=razon_cam_map)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].pie(camara.values, labels=camara.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Matrícula Cámara de Comercio (P1055)')
axes[1].pie(renov.values, labels=renov.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF5722'], startangle=90)
axes[1].set_title('Matrícula renovada (P661)')
rcs = razon_cam.sort_values(ascending=True)
axes[2].barh(rcs.index, rcs.values, color='#9C27B0')
axes[2].bar_label(axes[2].containers[0], labels=[f'{v:,.0f}' for v in rcs.values], padding=4)
axes[2].set_xlabel('Micronegocios estimados')
axes[2].set_title('Razón para no matricularse (P1057)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('Cámara de Comercio – Micronegocios Cesar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Declaraciones tributarias: Renta, IVA e ICA (P2991–P2993)

In [ ]:
tribut_vars = {'P2991': 'Renta', 'P2992': 'IVA', 'P2993': 'ICA'}
resumen = []
for var, etiqueta in tribut_vars.items():
    si  = cesar[cesar[var] == 1]['F_EXP'].sum()
    no  = cesar[cesar[var] == 2]['F_EXP'].sum()
    tot = si + no
    resumen.append({'Declaración': etiqueta, 'Sí presentó': si, 'No presentó': no, '% Sí': round(si/tot*100,1)})
df_trib = pd.DataFrame(resumen).set_index('Declaración')
print(df_trib)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_trib)); w = 0.35
b1 = ax.bar(x - w/2, df_trib['Sí presentó'], w, label='Sí presentó', color='#4CAF50')
b2 = ax.bar(x + w/2, df_trib['No presentó'], w, label='No presentó', color='#F44336')
ax.set_xticks(x); ax.set_xticklabels(df_trib.index)
ax.set_ylabel('Micronegocios estimados')
ax.set_title('Declaraciones tributarias – Micronegocios Cesar')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.bar_label(b1, labels=[f'{v:,.0f}' for v in df_trib['Sí presentó']], rotation=45, padding=3, fontsize=9)
ax.bar_label(b2, labels=[f'{v:,.0f}' for v in df_trib['No presentó']], rotation=45, padding=3, fontsize=9)
ax.legend(); plt.tight_layout(); plt.show()

## 7. Índice sintético de formalización (3 dimensiones)

In [ ]:
cesar['formal_rut']    = (cesar['P1633'] == 1).astype(int)
cesar['formal_cont']   = (cesar['P640']  == 1).astype(int)
cesar['formal_camara'] = (cesar['P1055'] == 1).astype(int)
cesar['indice_formal'] = cesar[['formal_rut','formal_cont','formal_camara']].sum(axis=1)

idx_dist = cesar.groupby('indice_formal')['F_EXP'].sum()
idx_pct  = (idx_dist / idx_dist.sum() * 100).round(1)
etiq = ['0 – Sin formalización','1 – Una dimensión','2 – Dos dimensiones','3 – Completamente formal']
cols = ['#F44336','#FF9800','#2196F3','#4CAF50']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(etiq[:len(idx_dist)], idx_dist.values, color=cols[:len(idx_dist)])
ax.bar_label(bars, labels=[f'{v:,.0f}\n({p}%)' for v, p in zip(idx_dist.values, idx_pct.values)], padding=4)
ax.set_ylabel('Micronegocios estimados')
ax.set_title('Índice sintético de formalización (0–3) – Cesar')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()

def tasa(df, var):
    return df[df[var] == 1]['F_EXP'].sum() / df['F_EXP'].sum() * 100

print('Tasa de formalización por dimensión:')
for var, nombre in [('formal_rut','RUT'),('formal_cont','Contabilidad'),('formal_camara','Cámara de Comercio')]:
    print(f'  {nombre}: {tasa(cesar, var):.1f}%')

## 8. Tipo de establecimiento (CLASE_TE)

In [ ]:
clase_map = {1:'Vivienda/parte de vivienda',2:'Local/oficina/bodega',
             3:'Vía pública/espacio abierto',4:'En obra/construcción',5:'Vehículo',6:'Otro'}
clase = cesar.groupby('CLASE_TE')['F_EXP'].sum().rename(index=clase_map).sort_values(ascending=False)
clase_pct = (clase / clase.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(clase.index[::-1], clase.values[::-1], color='#00897B')
ax.bar_label(bars, labels=[f'{v:,.0f} ({p}%)'
             for v, p in zip(clase.values[::-1], clase_pct.values[::-1])], padding=5)
ax.set_xlabel('Micronegocios estimados')
ax.set_title('Tipo de establecimiento – Micronegocios Cesar (CLASE_TE)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout(); plt.show()

## 9. Comparación Cesar vs. total nacional

In [ ]:
nacional = df_raw.copy()
nacional['formal_rut']    = (nacional['P1633'] == 1).astype(int)
nacional['formal_cont']   = (nacional['P640']  == 1).astype(int)
nacional['formal_camara'] = (nacional['P1055'] == 1).astype(int)

comparacion = pd.DataFrame({
    'Cesar':    [tasa(cesar,   'formal_rut'), tasa(cesar,   'formal_cont'), tasa(cesar,   'formal_camara')],
    'Nacional': [tasa(nacional,'formal_rut'), tasa(nacional,'formal_cont'), tasa(nacional,'formal_camara')]
}, index=['RUT','Registros contables','Cámara de Comercio']).round(1)
print(comparacion)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comparacion)); w = 0.35
b1 = ax.bar(x - w/2, comparacion['Cesar'],    w, label='Cesar',    color='#2196F3')
b2 = ax.bar(x + w/2, comparacion['Nacional'], w, label='Nacional', color='#9E9E9E')
ax.set_xticks(x); ax.set_xticklabels(comparacion.index)
ax.set_ylabel('Tasa de formalización (%)')
ax.set_title('Formalización: Cesar vs. Nacional')
ax.bar_label(b1, labels=[f'{v:.1f}%' for v in comparacion['Cesar']],    padding=3)
ax.bar_label(b2, labels=[f'{v:.1f}%' for v in comparacion['Nacional']], padding=3)
ax.set_ylim(0, 100); ax.legend(); plt.tight_layout(); plt.show()

## 10. Resumen ejecutivo

In [ ]:
univ       = cesar['F_EXP'].sum()
pct_rut    = tasa(cesar, 'formal_rut')
pct_cont   = tasa(cesar, 'formal_cont')
pct_camara = tasa(cesar, 'formal_camara')
pct_3dim   = cesar[cesar['indice_formal'] == 3]['F_EXP'].sum() / univ * 100
pct_0dim   = cesar[cesar['indice_formal'] == 0]['F_EXP'].sum() / univ * 100
pct_renta  = (cesar[cesar['P2991'] == 1]['F_EXP'].sum()
              / cesar[cesar['P2991'].isin([1,2])]['F_EXP'].sum() * 100)

print('=' * 62)
print('RESUMEN EJECUTIVO – MICRONEGOCIOS CESAR (EMICRON DANE)')
print('=' * 62)
print(f'Universo estimado de micronegocios:  {univ:>12,.0f}')
print('-' * 62)
print('DIMENSIÓN TRIBUTARIA')
print(f'  Con RUT registrado:                {pct_rut:>11.1f}%')
print(f'  Declararon renta (último año):     {pct_renta:>11.1f}%')
print('-' * 62)
print('DIMENSIÓN CONTABLE')
print(f'  Llevan registros contables:        {pct_cont:>11.1f}%')
print('-' * 62)
print('DIMENSIÓN LEGAL')
print(f'  Matriculados Cámara de Comercio:   {pct_camara:>11.1f}%')
print('-' * 62)
print('ÍNDICE SINTÉTICO (3 dimensiones)')
print(f'  Completamente formales (3/3):      {pct_3dim:>11.1f}%')
print(f'  Completamente informales (0/3):    {pct_0dim:>11.1f}%')
print('=' * 62)

---
# PARTE 2 – Sectores con Mayor Potencial Sostenible
## Análisis de Garantías FNG por CIIU (2024–2025)
**Fuente:** FNG – No. de Garantías por CIIU  
**Archivo:** Datos CIIU análisis GGGI.xlsx

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

RUTA_EXCEL = (r'C:\Users\Usuario\OneDrive - Global Green Growth Institute'
              r'\Documentos\2025\Outputs\Output5\3. Clasificación de destinos FNG'
              r'\Datos CIIU análisis GGGI.xlsx')
CESAR_COL = 13
TOTAL_COL = 34

def parse_pivot(xl, sheet, year):
    raw = xl.parse(sheet, header=None)
    records = []
    current_ciiu = None
    for _, row in raw.iloc[5:].iterrows():
        label = str(row.iloc[0]).strip()
        if label in ('nan', 'Total general', ''):
            continue
        total = pd.to_numeric(row.iloc[TOTAL_COL], errors='coerce')
        cesar = pd.to_numeric(row.iloc[CESAR_COL], errors='coerce')
        try:
            current_ciiu = int(float(label))
        except (ValueError, TypeError):
            if current_ciiu is not None:
                records.append({'CIIU': current_ciiu, 'Destino': label,
                                'Total_nacional': total, 'Cesar': cesar, 'Año': year})
    return pd.DataFrame(records)

xl  = pd.ExcelFile(RUTA_EXCEL, engine='openpyxl')
df24 = parse_pivot(xl, 'No garantías por CIIU 2024', 2024)
df25 = parse_pivot(xl, 'No garantías por CIIU 2025', 2025)
df   = pd.concat([df24, df25], ignore_index=True)
print(f'Registros 2024: {len(df24):,}  |  2025: {len(df25):,}')
print(f'CIIU únicos 2024: {df24.CIIU.nunique()}  |  2025: {df25.CIIU.nunique()}')
print('Destinos disponibles:', df.Destino.str.strip().unique())

## Taxonomía verde – definición de sectores con potencial sostenible

In [ ]:
CIIU_VERDE = {
    # Agricultura y naturaleza
    111:'Agricultura y naturaleza', 112:'Agricultura y naturaleza',
    113:'Agricultura y naturaleza', 114:'Agricultura y naturaleza',
    115:'Agricultura y naturaleza', 119:'Agricultura y naturaleza',
    121:'Agricultura y naturaleza', 122:'Agricultura y naturaleza',
    123:'Agricultura y naturaleza', 124:'Agricultura y naturaleza',
    125:'Agricultura y naturaleza', 126:'Agricultura y naturaleza',
    141:'Agricultura y naturaleza', 142:'Agricultura y naturaleza',
    143:'Agricultura y naturaleza', 144:'Agricultura y naturaleza',
    145:'Agricultura y naturaleza', 149:'Agricultura y naturaleza',
    150:'Agricultura y naturaleza',
    # Energía y agua
    3511:'Energía y agua', 3512:'Energía y agua', 3513:'Energía y agua', 3514:'Energía y agua',
    3520:'Energía y agua', 3530:'Energía y agua',
    3600:'Energía y agua', 3700:'Energía y agua',
    # Gestión de residuos
    3811:'Gestión de residuos', 3821:'Gestión de residuos',
    3830:'Gestión de residuos', 3900:'Gestión de residuos',
    # Construcción sostenible
    4111:'Construcción sostenible', 4112:'Construcción sostenible',
    4210:'Construcción sostenible', 4220:'Construcción sostenible', 4290:'Construcción sostenible',
    4311:'Construcción sostenible', 4312:'Construcción sostenible',
    4321:'Construcción sostenible', 4322:'Construcción sostenible',
    4329:'Construcción sostenible', 4330:'Construcción sostenible', 4390:'Construcción sostenible',
    # Transporte limpio
    4921:'Transporte limpio', 4922:'Transporte limpio', 4923:'Transporte limpio',
    5011:'Transporte limpio', 5021:'Transporte limpio', 5022:'Transporte limpio',
    # Investigación y tecnología
    7110:'Investigación y tecnología', 7120:'Investigación y tecnología',
    7210:'Investigación y tecnología', 7220:'Investigación y tecnología',
    # Educación
    8511:'Educación', 8512:'Educación', 8513:'Educación',
    8521:'Educación', 8522:'Educación', 8523:'Educación', 8530:'Educación',
    8541:'Educación', 8542:'Educación', 8543:'Educación', 8549:'Educación',
    8551:'Educación', 8552:'Educación', 8553:'Educación', 8559:'Educación', 8560:'Educación',
    # Salud
    8610:'Salud', 8621:'Salud', 8622:'Salud',
    8691:'Salud', 8692:'Salud', 8699:'Salud',
    # Silvicultura y paisajismo
    8130:'Silvicultura y paisajismo',
}

df['Macrosector'] = df['CIIU'].map(CIIU_VERDE)
df_verde = df[df['Macrosector'].notna()].copy()
df_verde['Destino_n'] = df_verde['Destino'].str.strip().replace({'Inversion Fija': 'Inversión Fija'})
df_verde['Cesar']     = pd.to_numeric(df_verde['Cesar'], errors='coerce').fillna(0)

print(f'Registros en sectores verdes: {len(df_verde):,}')
print(f'CIIU verdes encontrados: {sorted(df_verde.CIIU.unique())}')
print(df_verde.groupby('Año')['Total_nacional'].sum().map(lambda x: f'{x:,.0f}'))

## P2.1 – Garantías en sectores verdes: 2024 vs 2025 (nacional)

In [ ]:
macro_año = df_verde.groupby(['Macrosector','Año'])['Total_nacional'].sum().reset_index()
macros = macro_año['Macrosector'].unique()
x = np.arange(len(macros))
d24 = macro_año[macro_año['Año']==2024].set_index('Macrosector')['Total_nacional'].reindex(macros,fill_value=0)
d25 = macro_año[macro_año['Año']==2025].set_index('Macrosector')['Total_nacional'].reindex(macros,fill_value=0)
w = 0.38

fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x-w/2, d24.values, w, label='2024', color='#1976D2')
b2 = ax.bar(x+w/2, d25.values, w, label='2025', color='#43A047')
ax.set_xticks(x); ax.set_xticklabels(macros, rotation=25, ha='right')
ax.set_ylabel('No. de Garantías FNG')
ax.set_title('Garantías FNG en sectores con potencial sostenible – Nacional (2024 vs 2025)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.bar_label(b1, labels=[f'{v:,.0f}' for v in d24.values], rotation=45, padding=3, fontsize=8)
ax.bar_label(b2, labels=[f'{v:,.0f}' for v in d25.values], rotation=45, padding=3, fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()

var = ((d25-d24)/d24.replace(0,np.nan)*100).round(1)
print(pd.DataFrame({'2024':d24,'2025':d25,'Var%':var,'Total':d24+d25})
      .sort_values('Total',ascending=False).to_string())

## P2.2 – Destinos de crédito en sectores verdes

In [ ]:
dest_año = df_verde.groupby(['Destino_n','Año'])['Total_nacional'].sum().reset_index()
destinos = dest_año['Destino_n'].unique()
xd = np.arange(len(destinos))
dd24 = dest_año[dest_año['Año']==2024].set_index('Destino_n')['Total_nacional'].reindex(destinos,fill_value=0)
dd25 = dest_año[dest_año['Año']==2025].set_index('Destino_n')['Total_nacional'].reindex(destinos,fill_value=0)

fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(xd-w/2, dd24.values, w, label='2024', color='#1976D2')
b2 = ax.bar(xd+w/2, dd25.values, w, label='2025', color='#43A047')
ax.set_xticks(xd); ax.set_xticklabels(destinos, rotation=20, ha='right')
ax.set_ylabel('No. de Garantías FNG')
ax.set_title('Destinos de crédito en sectores verdes – Nacional')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.bar_label(b1, labels=[f'{v:,.0f}' for v in dd24.values], rotation=45, padding=3, fontsize=8)
ax.bar_label(b2, labels=[f'{v:,.0f}' for v in dd25.values], rotation=45, padding=3, fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()

tot  = dd24 + dd25
pct  = (tot/tot.sum()*100).round(1)
print(pd.DataFrame({'2024':dd24,'2025':dd25,'Total':tot,'% del total':pct})
      .sort_values('Total',ascending=False).to_string())

## P2.3 – Mapa de calor: macrosector × destino de crédito

In [ ]:
pivot_heat = (df_verde.groupby(['Macrosector','Destino_n'])['Total_nacional']
              .sum().unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot_heat.values, aspect='auto', cmap='YlGn')
ax.set_xticks(range(len(pivot_heat.columns)))
ax.set_xticklabels(pivot_heat.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot_heat.index)))
ax.set_yticklabels(pivot_heat.index)
plt.colorbar(im, ax=ax, label='No. garantías (2024+2025)')
for i in range(len(pivot_heat.index)):
    for j in range(len(pivot_heat.columns)):
        val = pivot_heat.values[i, j]
        ax.text(j, i, f'{val:,.0f}' if val > 0 else '', ha='center', va='center',
                fontsize=8, color='black' if val < pivot_heat.values.max()*0.6 else 'white')
ax.set_title('Garantías FNG por macrosector y destino de crédito (2024+2025)')
plt.tight_layout(); plt.show()

## P2.4 – Top 20 CIIU con mayor volumen en sectores verdes

In [ ]:
top_ciiu = (df_verde.groupby(['CIIU','Macrosector'])['Total_nacional'].sum()
            .reset_index().sort_values('Total_nacional',ascending=False).head(20))
top_ciiu['Etiqueta'] = top_ciiu['CIIU'].astype(str) + ' – ' + top_ciiu['Macrosector']

PAL = {'Agricultura y naturaleza':'#4CAF50','Educación':'#2196F3','Salud':'#E91E63',
       'Construcción sostenible':'#FF9800','Transporte limpio':'#9C27B0',
       'Energía y agua':'#00BCD4','Gestión de residuos':'#795548',
       'Investigación y tecnología':'#607D8B','Silvicultura y paisajismo':'#8BC34A'}
colors = [PAL.get(m,'#9E9E9E') for m in top_ciiu['Macrosector']]

fig, ax = plt.subplots(figsize=(11,7))
bars = ax.barh(top_ciiu['Etiqueta'][::-1], top_ciiu['Total_nacional'][::-1], color=colors[::-1])
ax.bar_label(bars, labels=[f'{v:,.0f}' for v in top_ciiu['Total_nacional'][::-1]], padding=4, fontsize=9)
ax.set_xlabel('No. de Garantías FNG (2024+2025)')
ax.set_title('Top 20 CIIU – Sectores con mayor potencial sostenible')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
from matplotlib.patches import Patch
leg = [Patch(facecolor=c, label=s) for s,c in PAL.items() if s in top_ciiu['Macrosector'].values]
ax.legend(handles=leg, loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

## P2.5 – Análisis Cesar: garantías en sectores verdes

In [ ]:
cesar_macro = df_verde.groupby(['Macrosector','Año'])['Cesar'].sum().reset_index()
macros_c = cesar_macro['Macrosector'].unique()
xc = np.arange(len(macros_c))
dc24 = cesar_macro[cesar_macro['Año']==2024].set_index('Macrosector')['Cesar'].reindex(macros_c,fill_value=0)
dc25 = cesar_macro[cesar_macro['Año']==2025].set_index('Macrosector')['Cesar'].reindex(macros_c,fill_value=0)

fig, ax = plt.subplots(figsize=(14,5))
b1 = ax.bar(xc-w/2, dc24.values, w, label='2024', color='#1976D2')
b2 = ax.bar(xc+w/2, dc25.values, w, label='2025', color='#43A047')
ax.set_xticks(xc); ax.set_xticklabels(macros_c, rotation=25, ha='right')
ax.set_ylabel('No. de Garantías FNG – Cesar')
ax.set_title('Garantías FNG en sectores verdes – Cesar (2024 vs 2025)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.bar_label(b1, labels=[f'{int(v)}' for v in dc24.values], rotation=45, padding=3, fontsize=8)
ax.bar_label(b2, labels=[f'{int(v)}' for v in dc25.values], rotation=45, padding=3, fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()

tot_cesar = dc24+dc25
tot_nac   = d24.reindex(macros_c,fill_value=0)+d25.reindex(macros_c,fill_value=0)
pct_cesar = (tot_cesar/tot_nac.replace(0,np.nan)*100).round(2)
print(pd.DataFrame({'Cesar 2024':dc24,'Cesar 2025':dc25,
                    'Cesar total':tot_cesar,'% del nacional':pct_cesar})
      .sort_values('Cesar total',ascending=False).to_string())

## P2.6 – Destinos transversales: Inversión Fija e Innovación

In [ ]:
destinos_clave = ['Inversión Fija','Innovación']
df_clave = df_verde[df_verde['Destino_n'].isin(destinos_clave)]
pivot_clave = (df_clave.groupby(['Macrosector','Destino_n'])['Total_nacional']
               .sum().unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(12,5))
xk = np.arange(len(pivot_clave))
palette = {'Inversión Fija':'#1976D2','Innovación':'#E91E63'}
cols_plot = [c for c in destinos_clave if c in pivot_clave.columns]
for i, col in enumerate(cols_plot):
    offset = (i-len(cols_plot)/2+0.5)*0.35
    bars = ax.bar(xk+offset, pivot_clave[col].values, 0.35, label=col, color=palette[col])
    ax.bar_label(bars, labels=[f'{v:,.0f}' for v in pivot_clave[col].values],
                 rotation=45, padding=3, fontsize=8)
ax.set_xticks(xk); ax.set_xticklabels(pivot_clave.index, rotation=25, ha='right')
ax.set_ylabel('No. de Garantías FNG (2024+2025)')
ax.set_title('Destinos transversales para la transición sostenible')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.legend(); plt.tight_layout(); plt.show()
print(pivot_clave.assign(Total=pivot_clave.sum(axis=1)).sort_values('Total',ascending=False).to_string())

## P2.7 – Resumen ejecutivo: potencial verde FNG

In [ ]:
tot24     = df_verde[df_verde['Año']==2024]['Total_nacional'].sum()
tot25     = df_verde[df_verde['Año']==2025]['Total_nacional'].sum()
tot_all   = df['Total_nacional'].sum()
pct_v     = (tot24+tot25)/tot_all*100
csar24    = df_verde[df_verde['Año']==2024]['Cesar'].sum()
csar25    = df_verde[df_verde['Año']==2025]['Cesar'].sum()
csar_all  = df['Cesar'].sum()
pct_cv    = (csar24+csar25)/csar_all*100 if csar_all>0 else 0
top24     = df_verde[df_verde['Año']==2024].groupby('Macrosector')['Total_nacional'].sum().idxmax()
top25     = df_verde[df_verde['Año']==2025].groupby('Macrosector')['Total_nacional'].sum().idxmax()
inv_fija  = df_verde[df_verde['Destino_n']=='Inversión Fija'].groupby('Año')['Total_nacional'].sum()

print('='*65)
print('RESUMEN EJECUTIVO – SECTORES VERDES FNG (GGGI)')
print('='*65)
print(f'Garantías en sectores verdes 2024:        {tot24:>12,.0f}')
print(f'Garantías en sectores verdes 2025:        {tot25:>12,.0f}')
print(f'% del total de garantías FNG:             {pct_v:>11.1f}%')
print('-'*65)
print(f'Sector líder 2024: {top24}')
print(f'Sector líder 2025: {top25}')
print('-'*65)
print('Inversión Fija en sectores verdes:')
for yr, val in inv_fija.items():
    print(f'  {yr}: {val:,.0f} garantías')
print('-'*65)
print(f'Garantías verdes Cesar 2024:              {csar24:>12,.0f}')
print(f'Garantías verdes Cesar 2025:              {csar25:>12,.0f}')
print(f'% Cesar / total nacional FNG:             {pct_cv:>11.1f}%')
print('='*65)
print('\nTop 5 CIIU con mayor potencial (2024+2025):')
top5 = (df_verde.groupby(['CIIU','Macrosector'])['Total_nacional'].sum()
        .reset_index().sort_values('Total_nacional',ascending=False).head(5))
for _,r in top5.iterrows():
    print(f'  CIIU {r.CIIU} ({r.Macrosector}): {r.Total_nacional:,.0f} garantías')